In [ ]:
# Name: Kushagra Agrawal
# A. No: 2026B01010412
# GitHub link: https://github.com/agrawalkushagra859-afk/Hotel_Booking_And_Occupancy_Analysis

In [ ]:
import numpy as np
import pandas as pd

# 1. Load data & remove duplicates safely
df = pd.read_csv('13_hotel_bookings.csv')
df = df.drop_duplicates(subset=['Booking_ID']).copy()

In [ ]:
# 2. Fill missing values using Median (best for skewed prices/days)
df['Lead_Time_Days'] = df['Lead_Time_Days'].fillna(
    df['Lead_Time_Days'].median()
)
df['Room_Rate'] = df['Room_Rate'].fillna(
    df.groupby('Room_Type')['Room_Rate'].transform('median')
)

In [ ]:
# 3. Extract Month from Date
df['Checkin_Date'] = pd.to_datetime(df['Checkin_Date'])
df['Month'] = df['Checkin_Date'].dt.month_name()

In [ ]:
# 4. Create Cancellation Flag & Realized Revenue (Vectorized if/else)
df['Is_Canceled_Flag'] = df['Is_Canceled'].isin([1, 'Yes', 'Canceled', True])
df['Revenue'] = np.where(
    ~df['Is_Canceled_Flag'], df['Nights'] * df['Room_Rate'], 0
)

In [ ]:
# 5. Filter Active Bookings (.loc = label filter, .iloc = position slice)
active = df.loc[~df['Is_Canceled_Flag']].copy()
print('First 3 rows:\n', active.iloc[:3, :4])

In [ ]:
# 6. GroupBy & Merge (SQL-style Left Join)
total_bk = df.groupby('Booking_Channel')['Booking_ID'].count().reset_index()
active_rev = active.groupby('Booking_Channel')['Revenue'].sum().reset_index()

channel_report = pd.merge(total_bk, active_rev, on='Booking_Channel', how='left')
print('\nChannel Report:\n', channel_report)

In [ ]:
# 7. Outlier Detection using IQR (25th and 75th percentile)
q1 = np.percentile(active['Revenue'], 25)
q3 = np.percentile(active['Revenue'], 75)
upper_limit = q3 + 1.5 * (q3 - q1)

outliers = active[active['Revenue'] > upper_limit]
print(f'\nHigh-Value Outliers (above {upper_limit:.0f}): {len(outliers)}')